In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

In [2]:
def load_data():
    return (
       pd.read_csv("data_raw/Characters.csv", sep=";"),
       pd.read_csv("data_raw/Potions.csv", sep=";"),
       pd.read_csv("data_raw/Spells.csv", sep=";"),
       pd.read_csv("data_raw/harry_potter_master_data.csv")
    )

# Load raw datasets
characters, potions, spells, master = load_data()

In [3]:
characters.info()
potions.info()
spells.info()
master.info()

characters.head()

<class 'pandas.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Id            140 non-null    int64
 1   Name          140 non-null    str  
 2   Gender        139 non-null    str  
 3   Job           121 non-null    str  
 4   House         101 non-null    str  
 5   Wand          132 non-null    str  
 6   Patronus      123 non-null    str  
 7   Species       140 non-null    str  
 8   Blood status  123 non-null    str  
 9   Hair colour   123 non-null    str  
 10  Eye colour    86 non-null     str  
 11  Loyalty       89 non-null     str  
 12  Skills        113 non-null    str  
 13  Birth         127 non-null    str  
 14  Death         42 non-null     str  
dtypes: int64(1), str(14)
memory usage: 16.5 KB
<class 'pandas.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             

,Id,Name,Gender,Job,House,Wand,Patronus,Species,Blood status,Hair colour,Eye colour,Loyalty,Skills,Birth,Death
0,1,Harry James Potter,Male,Student,Gryffindor,"11"" Holly phoenix feather",Stag,Human,Half-blood,Black,Bright green,Albus Dumbledore | Dumbledore's Army | Order o...,Parseltongue| Defence Against the Dark Arts | ...,31 July 1980,NaN
1,2,Ronald Bilius Weasley,Male,Student,Gryffindor,"12"" Ash unicorn tail hair",Jack Russell terrier,Human,Pure-blood,Red,Blue,Dumbledore's Army | Order of the Phoenix | Hog...,Wizard chess | Quidditch goalkeeping,1 March 1980,NaN
2,3,Hermione Jean Granger,Female,Student,Gryffindor,"10¾"" vine wood dragon heartstring",Otter,Human,Muggle-born,Brown,Brown,Dumbledore's Army | Order of the Phoenix | Hog...,Almost everything,"19 September, 1979",NaN
3,4,Albus Percival Wulfric Brian Dumbledore,Male,Headmaster,Gryffindor,"15"" Elder Thestral tail hair core",Phoenix,Human,Half-blood,Silver| formerly auburn,Blue,Dumbledore's Army | Order of the Phoenix | Hog...,Considered by many to be one of the most power...,Late August 1881,"30 June, 1997"
4,5,Rubeus Hagrid,Male,Keeper of Keys and Grounds | Professor of Care...,Gryffindor,"16"" Oak unknown core",NaN,Half-Human/Half-Giant,Part-Human (Half-giant),Black,Black,Albus Dumbledore | Order of the Phoenix | Hogw...,Resistant to stunning spells| above average st...,6 December 1928,NaN


In [4]:
def clean_columns(df):
    # standardize column names (lowercase, no spaces)
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
    return df

def clean_values(df):
    df = df.copy()

    obj_cols = df.select_dtypes(include=["object", "string"]).columns

    for col in obj_cols:
        df[col] = df[col].astype("string").str.strip()

        df[col] = df[col].replace(
            ["Unknown", "unknown", "", "NA", "N/A", "None"],
            np.nan
        )

    return df


def process(df):
    # apply full cleaning pipeline
    return clean_values(clean_columns(df))

In [5]:
# Clean all datasets
characters = process(characters)
potions = process(potions)
spells = process(spells)
master = process(master)

In [6]:
# Remove low-value / noisy columns
characters = characters.drop(columns=["hair_colour", "eye_colour", "wand"], errors="ignore")
potions = potions.drop(columns=["difficulty_level"], errors="ignore")
spells = spells.drop(columns=["light"], errors="ignore")

In [7]:
# Track origin of each row after merge
for name, df in zip(
    ["characters", "potions", "spells", "master"],
    [characters, potions, spells, master]
):
    df["source"] = name

In [8]:
# Combine all datasets into one table
all_data = pd.concat(
    [characters, potions, spells, master],
    ignore_index=True,
    sort=False
)

In [9]:
# Remove duplicate rows
all_data = all_data.drop_duplicates()

In [10]:
def eda(df):
    df.info()
    df.shape
    df.isnull().sum().sort_values(ascending=False)
    df.duplicated().sum()

    if "name" in df.columns:
        df["name"].is_unique
        df["name"].duplicated().sum()


# Quick dataset check after cleaning
eda(all_data)

<class 'pandas.DataFrame'>
RangeIndex: 6420 entries, 0 to 6419
Data columns (total 57 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 140 non-null    float64
 1   name               6413 non-null   string 
 2   gender             4859 non-null   string 
 3   job                121 non-null    string 
 4   house              1186 non-null   string 
 5   patronus           166 non-null    string 
 6   species            5422 non-null   string 
 7   blood_status       1532 non-null   string 
 8   loyalty            89 non-null     string 
 9   skills             113 non-null    string 
 10  birth              127 non-null    string 
 11  death              42 non-null     string 
 12  source             6420 non-null   str    
 13  known_ingredients  42 non-null     string 
 14  effect             855 non-null    string 
 15  characteristics    170 non-null    string 
 16  incantation        394 non-null    

In [11]:
# Save cleaned dataset
all_data.to_csv("data_processed/harry_potter_cleaned.csv", index=False)